# DATA IO + Cleaning

In [ ]:
# libraries
import pandas as pd
import IPython.display as display
import numpy as np
import os

# utils
from utils.data_io_clean import (
    load_and_clean_broker,
    load_and_clean_nse_eq_master,
    load_and_clean_nse_etf_master,
    load_and_clean_sgb,
)

from utils.dataset_builder import (
    build_canonical_portfolio,
    build_historical_price_dataset,
)

from utils.api_io import (
    fetch_historical_prices_n_years,
)

from utils.features import (
    compute_daily_ret,
    compute_annualized_mean_ret,
    compute_individual_annualized_volatility,
    winsorize_returns,
)

from utils.universe import (
    remove_securities,
    remove_securities_leq_weight_w,
)

# config
from src import config

np.random.seed(420)

## Loading Initial Data

In [ ]:
nse_eq_master_df = load_and_clean_nse_eq_master(path=config.NSE_EQ_PATH)
nse_etf_master_df = load_and_clean_nse_etf_master(path=config.NSE_ETF_PATH)
current_portfolio = load_and_clean_broker(path=config.PORTFOLIO_PATH)
sgb_price_df = load_and_clean_sgb(path=config.SGB_PATH)

display.display(nse_eq_master_df.tail(5))
display.display(nse_etf_master_df.tail(5))
display.display(sgb_price_df.tail(5))
display.display(current_portfolio)

## Build Primary Datasets

### Convert Broker Data into Canonical Format

In [ ]:
canon = build_canonical_portfolio(
    portfolio_df=current_portfolio,
    nse_eq_df=nse_eq_master_df,
    nse_etf_df=nse_etf_master_df,
    sgb_df=sgb_price_df,
)

canon

Now, we have a canonical dataframe that contains all the data from the broker that has been merged with the NSE masters. We have also derived the current prices and weights using the data fetched from yfinance. 

**Note:** We have ignored the single Mutual Fund for now because this is a very minor holding and is also actively managed by the AMC so it is beyond the scope of our current analysis.

### Build Historical Price Dataset (10Y)

In [ ]:
equities_3y_price_df = fetch_historical_prices_n_years(
    symbols=canon["symbol"].tolist(),
    n_years=10,
)

price_history_df = build_historical_price_dataset(
    eq_etf_historical=equities_3y_price_df,
    sgb_df=sgb_price_df,
)

price_history_df

### Analyze Historical Price Dataset

In [ ]:
price_history_df.describe()

In [ ]:
price_history_df.isna().sum()

In [ ]:
for col in price_history_df.columns:
    print(f"{str(col)}: {price_history_df[col].first_valid_index()}")

In [ ]:
print(canon[["symbol", "weight"]])

From the above we can see that some securities have short histories. However, trimming our data to get the first common date with actual data would cost us a majority of the data we have. Therefore we will try fixing this. MOSMALL250 has a very short history and is also a very insignificant holding so we can simply drop it and recompute the weights.

In [ ]:
mosmall_weight = canon[canon["symbol"] == "MOSMALL250.NS"]["weight"].iloc[0]
canon = canon[canon["symbol"] != "MOSMALL250.NS"].copy()
canon["weight"] /= 1 - mosmall_weight
canon

In [ ]:
canon["weight"].sum()

In [ ]:
price_history_df = price_history_df.drop(columns="MOSMALL250.NS")
price_history_df

In [ ]:
price_history_df.info

It is also known that MONIFTY500 is particularly heavy weighted. We can, however, proxy it using the returns of the TRI index it represents. We will have to drop any securities with similarly short or even shorter histories that can not be proxied.

Now that we have dropped MOSMALL250, we will do the following:
- Calculate daily returns.
- Fix any outliers if found.
- Calculate individual volatility using the full history.
- Calculate pairwise correlations.
- Use pairwise correlations and individual volatilities (composite estimator) to construct a covariance matrix since some securities have much smaller histories.
- Eigenvalue clipping to ensure PSD cov matrix

In [ ]:
ret_df = compute_daily_ret(price_history=price_history_df)

ret_df

Let's analyze these returns to see if there are any outliers.

In [ ]:
ret_df.describe()

Some securities do seem to have massive outliers judging by max returns over 800%. Let's also look at the annualized std.

In [ ]:
compute_individual_annualized_volatility(ret=ret_df)

We will fix the outliers using Winsorization with MAD. We will use a threshold of 8 to only catch huge deviations.

In [ ]:
ret_df = winsorize_returns(returns=ret_df, k=8)
ret_df

In [ ]:
ret_df.describe()

In [ ]:
# Test dropping
canon_1, ret_1 = remove_securities(
    canon_df=canon, ret=ret_df, symbols=["SGBMAY28", "NEXT50.NS"]
)
display.display(canon_1)
display.display(ret_1)

In [ ]:
# Load NIFTY 500 Index into a df and see its description
nifty_500_tri = pd.read_csv(os.path.join(config.PROCESSED_DATA_DIR, "NIFTY500.csv"))
nifty_500_tri

In [ ]:
nifty_500_tri.info()